Імпорт бібліотек

In [ ]:
import urllib.request
import os
import datetime
import pandas as pd
import re

Завантаження даних (urllib) із запобіганням колізій

In [ ]:
def download_vhi_data(out_dir="data"):
    if not os.path.exists(out_dir):
        os.makedirs(out_dir)
index_mapping = {
    1: 22, 2: 24, 3: 23, 4: 25, 5: 3, 6: 4, 7: 8, 8: 19, 9: 20, 10: 21,
    11: 9, 12: 26, 13: 10, 14: 11, 15: 12, 16: 13, 17: 14, 18: 15, 19: 16,
    20: 27, 21: 17, 22: 18, 23: 6, 24: 1, 25: 2, 26: 7, 27: 5
}

def load_and_clean_data(data_dir="data"):
    all_files = [os.path.join(data_dir, f) for f in os.listdir(data_dir) if f.endswith('.csv')]
    dfs = []
    
    for file in all_files:
        old_id = int(re.search(r'vhi_id_(\d+)_', file).group(1))
        
        df = pd.read_csv(file, index_col=False, header=1, skipinitialspace=True)
        
        df = df.dropna()
        df = df[~df['year'].astype(str).str.contains('<tt>')]
        
        df = df.rename(columns={'SMN': 'SMN', 'SMT': 'SMT', 'VCI': 'VCI', 'TCI': 'TCI', 'VHI': 'VHI'})
        
        for col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        df = df.dropna()
        
        new_id = index_mapping.get(old_id, old_id)
        df['Province_ID'] = new_id
        
        dfs.append(df)
        
    final_df = pd.concat(dfs, ignore_index=True)
    return final_df

df = load_and_clean_data()
print("Data cleaned. Shape:", df.shape)
    existing_files = os.listdir(out_dir)
    if len(existing_files) >= 27:
        print("Файли вже завантажено.")
        return

    now = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M")
    
    for province_id in range(1, 28):
        url = f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={province_id}&year1=1981&year2=2024&type=Mean"
        
        try:
            with urllib.request.urlopen(url) as response:
                html = response.read()
                
            filename = os.path.join(out_dir, f"vhi_id_{province_id}_{now}.csv")
            
            with open(filename, 'wb') as f:
                f.write(html)
            print(f"Завантажено: {filename}")
        except Exception as e:
            print(f"Помилка завантаження області {province_id}: {e}")

download_vhi_data()

Data Cleaning та зміна індексів

def get_vhi_by_year(df, province_id, year):
    """Ряд VHI для області за вказаний рік"""
    return df[(df['Province_ID'] == province_id) & (df['year'] == year)][['week', 'VHI']]

def get_vhi_range(df, province_id, start_year, end_year):
    """Ряд VHI за діапазон років для вказаної області"""
    return df[(df['Province_ID'] == province_id) & (df['year'] >= start_year) & (df['year'] <= end_year)][['year', 'week', 'VHI']]

def get_vhi_extremes(df, province_id, year):
    """Екстремуми для області та року"""
    subset = df[(df['Province_ID'] == province_id) & (df['year'] == year)]['VHI']
    return {
        'Min': subset.min(),
        'Max': subset.max(),
        'Mean': subset.mean(),
        'Median': subset.median()
    }

display(get_vhi_by_year(df, 1, 2020).head())
display(get_vhi_extremes(df, 1, 2020))